# 07 — RL agent (walk 1, PPO + cost-variant sweep)

PPO policy that allocates weights over the walk-1 ranker top-30 each Friday.
Trains separate policies at 4 transaction-cost levels (5/2/10/20 bps) so one
overnight run produces all the cost-sensitivity ablations for the paper.

**Spec:** `docs/superpowers/specs/2026-05-17-rl-agent-design.md`.

SB3 conventions: `check_env` validation → `EvalCallback` (best by val) →
`VecNormalize` (saved for backtest reuse) → TensorBoard. Single-walk MVP;
walks 2-16 deferred. Backtest = notebook 08.

## A. Setup

In [ ]:
from __future__ import annotations
import json, time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, ProgressBarCallback,
)

from src.utils.io import processed_dir, repo_root
from src.utils.ranker import friday_only, project_text_to_pca, load_walk_pca
from src.utils.rl_env import (
    PortfolioEnv,
    build_scoreboard_from_scored_panel,
)

WALK_ID = 1
TRAIN_START, TRAIN_END = '2002-01-01', '2007-12-31'
VAL_START,   VAL_END   = '2008-01-01', '2008-12-31'

PANEL_DIR  = processed_dir() / 'training_panel'
EMBED_DIR  = processed_dir() / 'finbert_stockday_embed'
RANKER_DIR = repo_root() / 'artifacts' / 'ranker' / f'walk-{WALK_ID:03d}'
OUT_ROOT   = repo_root() / 'artifacts' / 'rl' / f'walk-{WALK_ID:03d}'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Cost sweep — primary first; resumable across costs.
COSTS_BPS = [5, 2, 10, 20]
TOP_K = 30
EPISODE_LENGTH = 52
MAX_WEIGHT = 0.10
TOTAL_TIMESTEPS = 2_000_000
EVAL_FREQ = 10_000
CHECKPOINT_FREQ = 200_000
N_ENVS = 4
RANDOM_STATE = 42

print(f'walk {WALK_ID}; train {TRAIN_START}..{TRAIN_END}, val {VAL_START}..{VAL_END}')
print(f'cost sweep: {COSTS_BPS} bps; {TOTAL_TIMESTEPS:,} timesteps per variant')
print(f'out_root: {OUT_ROOT.relative_to(repo_root())}')

## B. Build the scoreboard

For each Friday in `[train_start, val_end]`, run the walk-1 ranker on every
stock in the universe, keep the top-30 by score. Persist to a parquet so
re-runs / kernel restarts don't re-score the panel.

In [ ]:
SCOREBOARD_PATH = OUT_ROOT / 'scoreboard.parquet'

if SCOREBOARD_PATH.exists():
    scoreboard = pd.read_parquet(SCOREBOARD_PATH)
    print(f'loaded scoreboard from disk: {len(scoreboard)} rows, '
          f'{scoreboard["date"].nunique()} Fridays')
else:
    ranker_bundle = joblib.load(RANKER_DIR / 'model.joblib')
    ranker_model = ranker_bundle['model']
    ranker_features = ranker_bundle['feature_names']
    pca, n_pca = load_walk_pca(WALK_ID)
    print(f'ranker loaded: {len(ranker_features)} features; PCA n_components={n_pca}')

    def _load_panel(start: str, end: str) -> pd.DataFrame:
        s, e = pd.Timestamp(start), pd.Timestamp(end)
        frames = []
        for y in range(s.year, e.year + 1):
            for p in sorted((PANEL_DIR / f'year={y}').glob('*.parquet')):
                df = pd.read_parquet(p)
                df['date'] = pd.to_datetime(df['date'])
                df = df[(df['date'] >= s) & (df['date'] <= e)]
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def _load_embed(start: str, end: str) -> pd.DataFrame:
        s, e = pd.Timestamp(start), pd.Timestamp(end)
        frames = []
        for y in range(s.year, e.year + 1):
            for p in sorted((EMBED_DIR / f'year={y}').glob('*.parquet')):
                df = pd.read_parquet(p, columns=['permno', 'date', 'vec'])
                df['date'] = pd.to_datetime(df['date'])
                df = df[(df['date'] >= s) & (df['date'] <= e)]
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    panel = _load_panel(TRAIN_START, VAL_END)
    embed = _load_embed(TRAIN_START, VAL_END)
    embed_pca = project_text_to_pca(embed, pca)

    # Friday-only + join PCA + drop rows w/o a forward return.
    fri = friday_only(panel).merge(embed_pca, on=['permno', 'date'], how='inner')
    fri = fri.dropna(subset=['fwd_ret_5d']).copy()

    # Build the feature matrix in the same column order the ranker expects.
    # Missing cols (shouldn't happen, but defensive) get NaN.
    X = pd.DataFrame({c: fri[c] if c in fri.columns else np.nan for c in ranker_features})
    fri['score'] = ranker_model.predict(X)

    scoreboard = build_scoreboard_from_scored_panel(fri, top_k=TOP_K)
    scoreboard.to_parquet(SCOREBOARD_PATH, compression='zstd', index=False)
    print(f'wrote scoreboard: {len(scoreboard)} rows, '
          f'{scoreboard["date"].nunique()} Fridays -> '
          f'{SCOREBOARD_PATH.relative_to(repo_root())}')

# Split into train / val views once.
scoreboard['date'] = pd.to_datetime(scoreboard['date'])
scoreboard_train = scoreboard[(scoreboard['date'] >= TRAIN_START) & (scoreboard['date'] <= TRAIN_END)].copy()
scoreboard_val   = scoreboard[(scoreboard['date'] >= VAL_START)   & (scoreboard['date'] <= VAL_END)].copy()
print(f'train Fridays: {scoreboard_train["date"].nunique()}; '
      f'val Fridays: {scoreboard_val["date"].nunique()}')